# Keyboard Actions

> Keyboard navigation library integration factories for token selector mode, actions, URL maps, and hidden action buttons.

In [ ]:
#| default_exp keyboard.actions

In [ ]:
#| export
from typing import Any, Dict, Tuple

from fasthtml.common import Div, Button, Hidden

from cjm_fasthtml_tailwind.utilities.layout import display_tw

from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.modes import KeyboardMode

from cjm_fasthtml_token_selector.core.config import TokenSelectorConfig
from cjm_fasthtml_token_selector.core.html_ids import TokenSelectorHtmlIds
from cjm_fasthtml_token_selector.js.core import global_callback_name

## Token Selector Mode

In [ ]:
#| export
def create_token_selector_mode(
    config:TokenSelectorConfig,         # config for this instance
    mode_name:str = "token-select",     # mode name for the keyboard nav system
    indicator_text:str = "Token Select", # mode indicator text
    exit_key:str = "",                  # exit key (empty = programmatic only via Escape KeyAction)
    exit_on_zone_change:bool = False,   # whether to exit on zone change
) -> KeyboardMode:  # configured keyboard mode
    """Create a keyboard mode that activates/deactivates the token selector."""
    return KeyboardMode(
        name=mode_name,
        on_enter=global_callback_name(config.prefix, "activate"),
        on_exit=global_callback_name(config.prefix, "deactivate"),
        indicator_text=indicator_text,
        exit_key=exit_key,
        exit_on_zone_change=exit_on_zone_change,
    )

In [ ]:
from cjm_fasthtml_token_selector.core.config import _reset_prefix_counter

_reset_prefix_counter()
cfg = TokenSelectorConfig(prefix="ts0")

mode = create_token_selector_mode(cfg)
assert mode.name == "token-select"
assert mode.on_enter == "ts0_activate"
assert mode.on_exit == "ts0_deactivate"
assert mode.indicator_text == "Token Select"
assert mode.exit_key == ""

# Custom mode name
mode2 = create_token_selector_mode(cfg, mode_name="split", indicator_text="Split Mode")
assert mode2.name == "split"
assert mode2.indicator_text == "Split Mode"

_reset_prefix_counter()
print("create_token_selector_mode tests passed!")

create_token_selector_mode tests passed!


## Navigation Actions

Non-movement keyboard actions. Movement keys (Left/Right) are handled by the
token selector's own key repeat engine, not by `KeyAction`.

In [ ]:
#| export
def create_token_nav_actions(
    config:TokenSelectorConfig,           # config for this instance
    zone_id:str,                          # focus zone ID
    mode_name:str = "token-select",       # mode name (must match the mode)
    confirm_button_id:str = "",           # HTMX button ID for confirm action
    cancel_button_id:str = "",            # HTMX button ID for cancel action
) -> Tuple[KeyAction, ...]:  # non-movement keyboard actions
    """Create keyboard actions for the token selector."""
    actions = []
    mode_scope = (mode_name,)

    # Confirm: Enter
    if confirm_button_id:
        actions.append(KeyAction(
            key="Enter",
            htmx_trigger=confirm_button_id,
            mode_names=mode_scope,
            mode_exit=True,
            description="Confirm selection",
            hint_group="Token Select",
            zone_ids=(zone_id,),
        ))
        # Space as alias (hidden from hints)
        actions.append(KeyAction(
            key=" ",
            htmx_trigger=confirm_button_id,
            mode_names=mode_scope,
            mode_exit=True,
            description="Confirm selection",
            hint_group="Token Select",
            show_in_hints=False,
            zone_ids=(zone_id,),
        ))

    # Cancel: Escape
    if cancel_button_id:
        actions.append(KeyAction(
            key="Escape",
            htmx_trigger=cancel_button_id,
            mode_names=mode_scope,
            mode_exit=True,
            description="Cancel",
            hint_group="Token Select",
            zone_ids=(zone_id,),
        ))

    # Home: move to first position
    actions.append(KeyAction(
        key="Home",
        js_callback=global_callback_name(config.prefix, "moveToFirst"),
        mode_names=mode_scope,
        description="Go to start",
        hint_group="Token Select",
        zone_ids=(zone_id,),
    ))

    # End: move to last position
    actions.append(KeyAction(
        key="End",
        js_callback=global_callback_name(config.prefix, "moveToLast"),
        mode_names=mode_scope,
        description="Go to end",
        hint_group="Token Select",
        zone_ids=(zone_id,),
    ))

    return tuple(actions)

In [ ]:
_reset_prefix_counter()
cfg = TokenSelectorConfig(prefix="ts0")

actions = create_token_nav_actions(
    cfg, zone_id="my-zone",
    confirm_button_id="confirm-btn",
    cancel_button_id="cancel-btn",
)

# Should have: Enter, Space, Escape, Home, End = 5 actions
assert len(actions) == 5

# No ArrowLeft/ArrowRight
keys = {a.key for a in actions}
assert "ArrowLeft" not in keys
assert "ArrowRight" not in keys
assert "Enter" in keys
assert "Escape" in keys
assert "Home" in keys
assert "End" in keys

# Home uses js_callback
home_action = [a for a in actions if a.key == "Home"][0]
assert home_action.js_callback == "ts0_moveToFirst"

# Without confirm/cancel
actions_min = create_token_nav_actions(cfg, zone_id="z")
assert len(actions_min) == 2  # Home + End only

_reset_prefix_counter()
print("create_token_nav_actions tests passed!")

create_token_nav_actions tests passed!


## URL Map and Action Buttons

In [ ]:
#| export
def build_token_selector_url_map(
    confirm_button_id:str,  # button ID for confirm action
    cancel_button_id:str,   # button ID for cancel action
    confirm_url:str,        # URL for confirm action
    cancel_url:str,         # URL for cancel action
) -> Dict[str, str]:  # button ID -> URL mapping
    """Build URL map for keyboard system with token selector action buttons."""
    url_map = {}
    if confirm_button_id and confirm_url:
        url_map[confirm_button_id] = confirm_url
    if cancel_button_id and cancel_url:
        url_map[cancel_button_id] = cancel_url
    return url_map

In [ ]:
#| export
def render_token_action_buttons(
    confirm_button_id:str,          # button ID for confirm
    cancel_button_id:str,           # button ID for cancel
    confirm_url:str,                # URL for confirm action
    cancel_url:str,                 # URL for cancel action
    ids:TokenSelectorHtmlIds,       # HTML IDs (for hx_include)
    extra_include:str = "",         # additional hx_include selectors
) -> Any:  # Div containing hidden action buttons
    """Render hidden HTMX buttons for confirm/cancel actions."""
    include = f"#{ids.anchor_input}, #{ids.focus_input}"
    if extra_include:
        include = f"{include}, {extra_include}"

    buttons = []
    if confirm_button_id and confirm_url:
        buttons.append(Button(
            id=confirm_button_id,
            hx_post=confirm_url,
            hx_swap="none",
            hx_include=include,
            cls=str(display_tw.hidden),
        ))
    if cancel_button_id and cancel_url:
        buttons.append(Button(
            id=cancel_button_id,
            hx_post=cancel_url,
            hx_swap="none",
            hx_include=include,
            cls=str(display_tw.hidden),
        ))

    return Div(*buttons, cls=str(display_tw.hidden))

In [ ]:
ids = TokenSelectorHtmlIds(prefix="ts0")

url_map = build_token_selector_url_map("confirm-btn", "cancel-btn", "/confirm", "/cancel")
assert url_map["confirm-btn"] == "/confirm"
assert url_map["cancel-btn"] == "/cancel"
print("build_token_selector_url_map test passed!")

btns = render_token_action_buttons("confirm-btn", "cancel-btn", "/confirm", "/cancel", ids)
assert btns is not None
print("render_token_action_buttons test passed!")

build_token_selector_url_map test passed!
render_token_action_buttons test passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()